# Indian Mutual Fund Portfolio Overlap Analyzer
### By Daksh Malhotra

**Objective:** Millions of Indian investors hold 3-5 mutual funds thinking they're diversified, but most funds hold the same stocks. This project analyzes portfolio overlap across 45 top Indian equity mutual funds to uncover hidden concentration risks.

**Data Source:** Scraped from Moneycontrol.com (publicly available portfolio holdings)

**Tools:** Python (requests, BeautifulSoup, pandas), SQL (SQLite)

## Data Collection

The scraper lives in [`scrape.py`](../scrape.py) at the repo root, deliberately kept out of
this notebook so the analysis below stays reproducible: everything from here on reads only
the committed database and never touches the network.

**Fund selection** — the top 10 funds by AUM in each of Large Cap, Mid Cap, Small Cap and
Flexi Cap, plus 5 index funds (Nifty 50 and Sensex trackers). 45 funds in total, covering
the most widely held equity schemes in India. Their URLs are stored in
`data/funds_list.json`, which `scrape.py` reads as its input.

**Two things the scrape had to handle:**

1. Moneycontrol renders *exited* holdings — stocks a fund has already sold — in the same
   table as current ones, hidden with `display: none`. Nothing raises if you include them.
   You simply get a plausible-looking row count and quietly wrong overlap numbers, which is
   far more dangerous than a crash. The scraper skips any row carrying that style.
2. Rows carry `-` and `#` markers indicating position changes, sitting in the same cell as
   the stock name and separated by a newline. Left in, `"# HDFC Bank Ltd."` and
   `"HDFC Bank Ltd."` are two different strings, so the shared holding vanishes from every
   `GROUP BY` and the overlap is understated. Stock names are stripped before use.

**A snapshot, not a live feed.** The data was captured in Feb/Mar 2026 and is committed to
the repo as `data/mutual_fund_holdings.csv` and `data/mutual_fund_overlap.db`. Moneycontrol
has since changed its portfolio-holdings URLs, so `scrape.py` no longer runs against the
live site. Freezing the dataset is deliberate rather than unfortunate: holdings change every
month, and a fixed snapshot is what keeps this notebook, the dashboard and the README
consistent with one another.

## Load the dataset

Everything below queries `data/mutual_fund_overlap.db`, built from the snapshot described
above. No network access is involved, so the notebook runs top to bottom from a fresh clone.

In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/mutual_fund_overlap.db')
print(f"Connected! Rows: {pd.read_sql('SELECT COUNT(*) FROM holdings', conn).iloc[0,0]}")

Connected! Rows: 3421


### Which stocks appear in the most mutual funds?
If the same stock is held by 30+ funds, any investor holding multiple funds is unknowingly over-exposed to that stock. Let's find out which stocks dominate Indian mutual fund portfolios.

In [9]:
query = """
SELECT 
    stock_name,
    COUNT(DISTINCT fund_name) as num_funds,
    ROUND(AVG(holding_pct), 2) as avg_holding_pct
FROM holdings
WHERE holding_pct > 0
GROUP BY stock_name
HAVING num_funds > 1
ORDER BY num_funds DESC
LIMIT 20
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                             stock_name  num_funds  avg_holding_pct
                           Eternal Ltd.         32             1.90
                        ICICI Bank Ltd.         30             6.61
                         HDFC Bank Ltd.         29             7.30
                         Axis Bank Ltd.         29             3.23
               Kotak Mahindra Bank Ltd.         26             2.69
               Reliance Industries Ltd.         25             5.05
                           Infosys Ltd.         25             3.20
                     Bharti Airtel Ltd.         25             3.69
                    State Bank Of India         24             3.67
                   Larsen & Toubro Ltd.         24             3.93
               Maruti Suzuki India Ltd.         22             2.24
               Mahindra & Mahindra Ltd.         21             2.44
     Sun Pharmaceutical Industries Ltd.         20             1.33
National Thermal Power Corporation Ltd.         

### Which fund pairs have the highest portfolio overlap?
This is the core question — if I hold Fund A and Fund B, how similar are their portfolios? I'm using weighted overlap (sum of minimum weights of common stocks), which is the same formula used by professional tools like Dezerv and PrimeInvestor.

In [10]:
query = """
WITH fund_pairs AS (
    SELECT 
        a.fund_name as fund_1,
        b.fund_name as fund_2,
        COUNT(DISTINCT a.stock_name) as common_stocks,
        ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as weighted_overlap
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name < b.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
    GROUP BY a.fund_name, b.fund_name
)
SELECT 
    fund_1,
    fund_2,
    common_stocks,
    weighted_overlap
FROM fund_pairs
ORDER BY weighted_overlap DESC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                              fund_1                               fund_2  common_stocks  weighted_overlap
            HDFC Nifty 50 Index Fund                 SBI Nifty Index Fund             50            100.00
            HDFC Nifty 50 Index Fund              UTI Nifty 50 Index Fund             50            100.00
                SBI Nifty Index Fund              UTI Nifty 50 Index Fund             50            100.00
            HDFC Nifty 50 Index Fund ICICI Prudential Nifty 50 Index Fund             50             99.76
ICICI Prudential Nifty 50 Index Fund                 SBI Nifty Index Fund             50             99.76
ICICI Prudential Nifty 50 Index Fund              UTI Nifty 50 Index Fund             50             99.76
          HDFC BSE Sensex Index Fund             HDFC Nifty 50 Index Fund             30             83.73
          HDFC BSE Sensex Index Fund                 SBI Nifty Index Fund             30             83.73
          HDFC BSE Sensex Index Fund 

### Which fund pairs have the LEAST overlap?
The flip side — these are the best fund combinations for diversification. If two funds share almost no stocks, holding both gives you genuine diversification.

In [11]:
query = """
WITH fund_pairs AS (
    SELECT 
        a.fund_name as fund_1,
        b.fund_name as fund_2,
        COUNT(DISTINCT a.stock_name) as common_stocks,
        ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as weighted_overlap
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name < b.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
    GROUP BY a.fund_name, b.fund_name
)
SELECT 
    fund_1,
    fund_2,
    common_stocks,
    weighted_overlap
FROM fund_pairs
ORDER BY weighted_overlap ASC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                       fund_1                          fund_2  common_stocks  weighted_overlap
  Parag Parikh Flexi Cap Fund                 SBI Midcap Fund              3              0.04
            Kotak Midcap Fund     Parag Parikh Flexi Cap Fund              3              0.15
         Quant Small Cap Fund                 SBI Midcap Fund              2              0.23
           DSP Small Cap Fund     Parag Parikh Flexi Cap Fund              2              0.28
  Parag Parikh Flexi Cap Fund           Sundaram Mid Cap Fund              7              0.33
          HSBC Small Cap Fund ICICI Prudential Large Cap Fund              1              0.34
          HDFC Small Cap Fund     Parag Parikh Flexi Cap Fund              4              0.36
 Canara Robeco Flexi Cap Fund              SBI Small Cap Fund              2              0.40
           DSP Small Cap Fund              UTI Large Cap Fund              1              0.40
          HDFC Large Cap Fund             HSBC Sma

### Most "crowded" stocks — where is mutual fund money concentrated?
Beyond just counting appearances, I wanted to see which stocks have the most total money flowing into them across all funds. A stock held by 25 funds at 7% each represents massive concentration risk for the market.

In [12]:
query = """
SELECT 
    stock_name,
    COUNT(DISTINCT fund_name) as num_funds,
    ROUND(AVG(holding_pct), 2) as avg_weight,
    ROUND(MAX(holding_pct), 2) as max_weight,
    ROUND(SUM(holding_pct), 2) as total_weight_across_funds
FROM holdings
WHERE holding_pct > 0
GROUP BY stock_name
ORDER BY total_weight_across_funds DESC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

              stock_name  num_funds  avg_weight  max_weight  total_weight_across_funds
          HDFC Bank Ltd.         29        7.30       14.07                     211.66
         ICICI Bank Ltd.         30        6.61       10.27                     198.32
Reliance Industries Ltd.         25        5.05        9.83                     126.18
    Larsen & Toubro Ltd.         24        3.93        6.34                      94.25
          Axis Bank Ltd.         29        3.23        7.44                      93.70
      Bharti Airtel Ltd.         25        3.69        5.59                      92.17
     State Bank Of India         24        3.67        5.26                      87.97
            Infosys Ltd.         25        3.20        4.88                      79.88
Kotak Mahindra Bank Ltd.         26        2.69        5.34                      69.97
            Eternal Ltd.         32        1.90        6.40                      60.89
Mahindra & Mahindra Ltd.         21        

### Which sectors dominate Indian mutual fund portfolios?
If most funds are overweight on the same sectors, a sector-specific downturn (like the 2018 NBFC crisis) would hit almost every mutual fund investor simultaneously.

In [13]:
query = """
SELECT 
    sector,
    COUNT(DISTINCT stock_name) as unique_stocks,
    COUNT(DISTINCT fund_name) as funds_invested,
    ROUND(AVG(holding_pct), 2) as avg_weight,
    ROUND(SUM(holding_pct), 2) as total_weight
FROM holdings
WHERE holding_pct > 0
GROUP BY sector
ORDER BY total_weight DESC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                                  sector  unique_stocks  funds_invested  avg_weight  total_weight
                     Private sector bank             15              45        3.89        665.63
       Computers - software & consulting             20              45        1.52        240.47
                         Pharmaceuticals             54              42        0.94        205.81
            Auto components & equipments             44              36        0.97        153.99
                  Refineries & marketing              4              35        3.27        147.15
    Non banking financial company (nbfc)             20              42        1.42        145.24
                      Civil construction             24              37        1.64        133.20
                      Public sector bank              8              36        2.54        121.96
       Passenger cars & utility vehicles              4              25        1.95        112.93
Telecom - cellular &

### Which funds are most concentrated (least diversified internally)?
Fewer stocks = higher conviction but higher risk. I wanted to see which funds run concentrated portfolios.

In [14]:
query = """
SELECT 
    fund_name,
    COUNT(stock_name) as total_stocks,
    ROUND(SUM(CASE WHEN holding_pct >= 5 THEN holding_pct ELSE 0 END), 2) as weight_in_top_holdings,
    ROUND(MAX(holding_pct), 2) as largest_single_stock,
    ROUND(AVG(holding_pct), 2) as avg_holding
FROM holdings
WHERE holding_pct > 0
GROUP BY fund_name
ORDER BY total_stocks ASC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                           fund_name  total_stocks  weight_in_top_holdings  largest_single_stock  avg_holding
        Motilal Oswal Flexi Cap Fund            23                   52.27                  8.93         4.02
           Motilal Oswal Midcap Fund            26                   56.19                  7.64         3.77
          HDFC BSE Sensex Index Fund            30                   50.05                 14.07         3.33
                  SBI Large Cap Fund            45                   26.65                  8.20         2.12
                 HDFC Large Cap Fund            46                   29.01                  9.50         2.14
            HDFC Nifty 50 Index Fund            50                   28.61                 11.83         2.00
ICICI Prudential Nifty 50 Index Fund            50                   28.54                 11.80         2.00
                SBI Nifty Index Fund            50                   28.61                 11.83         2.00
          

### Ranking top holdings within each fund using Window Functions
Using RANK() to see each fund's top stock picks ordered by weight. This reveals which stocks each fund manager is betting on the most.

In [15]:
query = """
SELECT 
    fund_name,
    stock_name,
    holding_pct,
    RANK() OVER (PARTITION BY fund_name ORDER BY holding_pct DESC) as rank
FROM holdings
WHERE holding_pct > 0
"""

result = pd.read_sql_query(query, conn)

# Show top 3 holdings per fund for first 5 funds
for fund in result['fund_name'].unique()[:5]:
    print(f"\n{fund}:")
    fund_data = result[result['fund_name'] == fund].head(3)
    for _, row in fund_data.iterrows():
        print(f"  #{int(row['rank'])} {row['stock_name'][:35]} → {row['holding_pct']}%")


Aditya Birla Sun Life Flexi Cap Fund:
  #1 ICICI Bank Ltd. → 6.48%
  #2 HDFC Bank Ltd. → 3.89%
  #3 Kotak Mahindra Bank Ltd. → 3.8%

Aditya Birla Sun Life Large Cap Fund:
  #1 ICICI Bank Ltd. → 7.49%
  #2 HDFC Bank Ltd. → 7.25%
  #3 Reliance Industries Ltd. → 4.77%

Axis Large Cap Fund:
  #1 ICICI Bank Ltd. → 8.73%
  #2 HDFC Bank Ltd. → 8.3%
  #3 Reliance Industries Ltd. → 6.06%

Axis Midcap Fund:
  #1 Federal Bank Ltd. → 3.98%
  #2 Fortis Healthcare Ltd. → 3.85%
  #3 Multi Commodity Exchange Of India L → 2.73%

Axis Small Cap Fund:
  #1 Krishna Institute of Medical Scienc → 2.9%
  #2 Multi Commodity Exchange Of India L → 2.82%
  #3 CCL Products (India) Ltd. → 2.69%


### How many stocks does it take to reach 50% of a fund's portfolio?
Using cumulative SUM as a window function. If just 5-6 stocks make up 50% of the fund, it's highly concentrated. If it takes 20+ stocks, the fund is well-diversified internally.

In [ ]:
query = """
WITH ranked AS (
    SELECT
        fund_name,
        stock_name,
        holding_pct,
        ROW_NUMBER() OVER (PARTITION BY fund_name
                           ORDER BY holding_pct DESC)          as stock_rank,
        SUM(holding_pct) OVER (PARTITION BY fund_name
                               ORDER BY holding_pct DESC
                               ROWS BETWEEN UNBOUNDED PRECEDING
                                        AND CURRENT ROW)       as cumulative_weight
    FROM holdings
    WHERE holding_pct > 0
)
SELECT
    fund_name,
    MIN(stock_rank) as stocks_to_50pct
FROM ranked
WHERE cumulative_weight >= 50
GROUP BY fund_name
ORDER BY stocks_to_50pct ASC
"""

result = pd.read_sql_query(query, conn)
print("Stocks needed to reach 50% of the portfolio (most concentrated first):\n")
for _, row in result.iterrows():
    print(f"  {row['fund_name'][:45]:45s} -> {row['stocks_to_50pct']} stocks")

Stocks needed to reach 50% of the portfolio (most concentrated first):

  HDFC BSE Sensex Index Fund                    -> 6 stocks
  Motilal Oswal Flexi Cap Fund                  -> 8 stocks
  Motilal Oswal Midcap Fund                     -> 8 stocks
  HDFC Nifty 50 Index Fund                      -> 9 stocks
  ICICI Prudential Large Cap Fund               -> 9 stocks
  ICICI Prudential Nifty 50 Index Fund          -> 9 stocks
  SBI Nifty Index Fund                          -> 9 stocks
  UTI Nifty 50 Index Fund                       -> 9 stocks
  Axis Large Cap Fund                           -> 10 stocks
  HDFC Large Cap Fund                           -> 10 stocks
  Canara Robeco Large Cap Fund                  -> 11 stocks
  HDFC Flexi Cap Fund                           -> 11 stocks
  Mirae Asset Large Cap Fund                    -> 11 stocks
  Parag Parikh Flexi Cap Fund                   -> 11 stocks
  SBI Large Cap Fund                            -> 11 stocks
  UTI Large Cap Fund 

### Real-world scenario: What happens if I hold both HDFC Large Cap and ICICI Large Cap?
Many investors hold 2-3 large cap funds thinking they're diversified. Let's see the actual combined exposure — how much of your money ends up in the same stocks?

In [ ]:
query = """
SELECT
    stock_name,
    ROUND(SUM(CASE WHEN fund_name = 'HDFC Large Cap Fund'
                   THEN holding_pct ELSE 0 END), 2)              as hdfc_large_cap,
    ROUND(SUM(CASE WHEN fund_name = 'ICICI Prudential Large Cap Fund'
                   THEN holding_pct ELSE 0 END), 2)              as icici_large_cap,
    ROUND(SUM(holding_pct), 2)                                   as combined_exposure
FROM holdings
WHERE fund_name IN ('HDFC Large Cap Fund', 'ICICI Prudential Large Cap Fund')
    AND holding_pct > 0
GROUP BY stock_name
ORDER BY combined_exposure DESC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                             stock_name  hdfc_large_cap  icici_large_cap  combined_exposure
                        ICICI Bank Ltd.            9.50             9.42              18.92
                         HDFC Bank Ltd.            8.58             9.16              17.74
               Reliance Industries Ltd.            4.96             5.98              10.94
                     Bharti Airtel Ltd.            5.59             4.11               9.70
                         Axis Bank Ltd.            3.70             4.61               8.31
                   Larsen & Toubro Ltd.            1.40             6.34               7.74
                           Infosys Ltd.            3.67             3.51               7.18
               Kotak Mahindra Bank Ltd.            5.34             0.00               5.34
               Maruti Suzuki India Ltd.            1.33             3.89               5.22
National Thermal Power Corporation Ltd.            2.13             2.55        

### Do large cap funds overlap more than small cap funds?
My hypothesis: Large cap funds should overlap more because SEBI mandates them to invest 80%+ in only ~100 stocks. Small cap funds have 1000+ options. Let's test this.

In [18]:
query = """
WITH fund_categories AS (
    SELECT fund_name,
        CASE 
            WHEN fund_name LIKE '%Large Cap%' THEN 'Large Cap'
            WHEN fund_name LIKE '%Mid Cap%' OR fund_name LIKE '%Midcap%' THEN 'Mid Cap'
            WHEN fund_name LIKE '%Small Cap%' THEN 'Small Cap'
            WHEN fund_name LIKE '%Flexi%' OR fund_name LIKE '%Flexicap%' THEN 'Flexi Cap'
            WHEN fund_name LIKE '%Index%' OR fund_name LIKE '%Sensex%' OR fund_name LIKE '%Nifty%' THEN 'Index'
            ELSE 'Other'
        END as category
    FROM holdings
    GROUP BY fund_name
),
overlap AS (
    SELECT 
        c1.category as category,
        a.fund_name as fund_1,
        b.fund_name as fund_2,
        COUNT(DISTINCT a.stock_name) as common_stocks,
        ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as weighted_overlap
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name < b.fund_name
    JOIN fund_categories c1 ON a.fund_name = c1.fund_name
    JOIN fund_categories c2 ON b.fund_name = c2.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
        AND c1.category = c2.category
    GROUP BY a.fund_name, b.fund_name
)
SELECT 
    category,
    COUNT(*) as fund_pairs,
    ROUND(AVG(common_stocks), 1) as avg_common_stocks,
    ROUND(AVG(weighted_overlap), 2) as avg_overlap_pct,
    ROUND(MAX(weighted_overlap), 2) as max_overlap_pct
FROM overlap
GROUP BY category
ORDER BY avg_overlap_pct DESC
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

 category  fund_pairs  avg_common_stocks  avg_overlap_pct  max_overlap_pct
    Index          10               42.0            93.40           100.00
Large Cap          45               29.6            57.12            72.98
Flexi Cap          45               17.0            32.46            49.60
  Mid Cap          44               24.8            27.63            48.05
Small Cap          45               22.0            12.08            26.79


### Zero overlap fund pairs — can two mid cap funds share absolutely no stocks?
While analyzing the category overlap data, I noticed one mid cap pair was missing from the results. This query finds which two funds have zero common stocks — despite being in the same category.

In [19]:
query = """
WITH midcap_funds AS (
    SELECT DISTINCT fund_name 
    FROM holdings 
    WHERE fund_name LIKE '%Mid Cap%' OR fund_name LIKE '%Midcap%'
),
all_pairs AS (
    SELECT a.fund_name as fund_1, b.fund_name as fund_2
    FROM midcap_funds a
    CROSS JOIN midcap_funds b
    WHERE a.fund_name < b.fund_name
),
overlap_pairs AS (
    SELECT 
        a.fund_name as fund_1, 
        b.fund_name as fund_2
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name < b.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
    GROUP BY a.fund_name, b.fund_name
)
SELECT ap.fund_1, ap.fund_2
FROM all_pairs ap
LEFT JOIN overlap_pairs op 
    ON ap.fund_1 = op.fund_1 AND ap.fund_2 = op.fund_2
WHERE op.fund_1 IS NULL
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                   fund_1          fund_2
Motilal Oswal Midcap Fund SBI Midcap Fund


### Which category combinations give the best diversification?
If I want to hold one fund from two different categories, which combination gives me the least overlap? This helps build an optimal multi-category portfolio.

In [ ]:
query = """
WITH fund_cat AS (
    SELECT fund_name,
        CASE 
            WHEN fund_name LIKE '%Large Cap%' THEN 'Large Cap'
            WHEN fund_name LIKE '%Mid Cap%' OR fund_name LIKE '%Midcap%' THEN 'Mid Cap'
            WHEN fund_name LIKE '%Small Cap%' THEN 'Small Cap'
            WHEN fund_name LIKE '%Flexi%' OR fund_name LIKE '%Flexicap%' THEN 'Flexi Cap'
            WHEN fund_name LIKE '%Index%' OR fund_name LIKE '%Sensex%' OR fund_name LIKE '%Nifty%' THEN 'Index'
            ELSE 'Other'
        END as category
    FROM holdings
    GROUP BY fund_name
),
pairs AS (
    SELECT 
        c1.category as cat_1,
        c2.category as cat_2,
        ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as overlap
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name < b.fund_name
    JOIN fund_cat c1 ON a.fund_name = c1.fund_name
    JOIN fund_cat c2 ON b.fund_name = c2.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
        AND c1.category != c2.category
    GROUP BY a.fund_name, b.fund_name
)
SELECT 
    MIN(cat_1, cat_2) || ' + ' || MAX(cat_1, cat_2) as combination,
    ROUND(AVG(overlap), 2) as avg_overlap,
    ROUND(MIN(overlap), 2) as min_overlap,
    COUNT(*) as pairs
FROM pairs
GROUP BY MIN(cat_1, cat_2), MAX(cat_1, cat_2)
ORDER BY avg_overlap ASC
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

          combination  avg_overlap  min_overlap  pairs
      Index + Mid Cap         4.29         0.94     50
Flexi Cap + Small Cap         4.89         0.28     97
Large Cap + Small Cap         5.05         0.34     88
    Index + Small Cap         5.57         0.88     35
  Mid Cap + Small Cap         6.42         0.23     97
  Large Cap + Mid Cap         7.72         2.34    100
  Flexi Cap + Mid Cap        10.13         0.04    100
Flexi Cap + Large Cap        40.35        17.09    100
    Flexi Cap + Index        42.90        25.52     50
    Index + Large Cap        63.22        50.59     50


### Hidden gems — high conviction stocks held by very few funds
These are stocks where a fund manager has placed a big bet (>2% weight) but very few other funds hold it. These represent unique, differentiated positions — potentially the fund manager's best ideas.

In [21]:
query = """
SELECT 
    stock_name,
    COUNT(DISTINCT fund_name) as num_funds,
    ROUND(MAX(holding_pct), 2) as max_weight,
    GROUP_CONCAT(fund_name, ' | ') as held_by
FROM holdings
WHERE holding_pct > 2
GROUP BY stock_name
HAVING num_funds <= 3
ORDER BY max_weight DESC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                            stock_name  num_funds  max_weight                                                                          held_by
                TVS Motor Company Ltd.          1        9.68                                                   ICICI Prudential Flexicap Fund
               Nifty 50 : Futures Near          3        8.93   Motilal Oswal Midcap Fund | Axis Small Cap Fund | Motilal Oswal Flexi Cap Fund
            One 97 Communications Ltd.          1        7.64                                                        Motilal Oswal Midcap Fund
           Kalyan Jewellers India Ltd.          3        7.48 Motilal Oswal Midcap Fund | Sundaram Mid Cap Fund | Motilal Oswal Flexi Cap Fund
CG Power and Industrial Solutions Ltd.          1        7.05                                                     Motilal Oswal Flexi Cap Fund
  Power Grid Corporation of India Ltd.          2        6.92                                Parag Parikh Flexi Cap Fund | HDFC Flexi Cap Fund

### Which fund house (AMC) dominates each sector?
Different AMCs may have sector biases based on their research teams' expertise. This reveals which AMC is the heaviest investor in each sector across all their funds.

In [22]:
query = """
WITH fund_amc AS (
    SELECT fund_name,
        CASE
            WHEN fund_name LIKE '%HDFC%' THEN 'HDFC'
            WHEN fund_name LIKE '%SBI%' THEN 'SBI'
            WHEN fund_name LIKE '%ICICI%' THEN 'ICICI'
            WHEN fund_name LIKE '%Axis%' THEN 'Axis'
            WHEN fund_name LIKE '%Kotak%' THEN 'Kotak'
            WHEN fund_name LIKE '%Nippon%' THEN 'Nippon'
            WHEN fund_name LIKE '%Mirae%' THEN 'Mirae'
            WHEN fund_name LIKE '%Motilal%' THEN 'Motilal Oswal'
            WHEN fund_name LIKE '%DSP%' THEN 'DSP'
            WHEN fund_name LIKE '%UTI%' THEN 'UTI'
            WHEN fund_name LIKE '%Parag%' THEN 'PPFAS'
            WHEN fund_name LIKE '%Canara%' THEN 'Canara Robeco'
            WHEN fund_name LIKE '%Aditya%' THEN 'Aditya Birla'
            WHEN fund_name LIKE '%Franklin%' THEN 'Franklin'
            WHEN fund_name LIKE '%Tata%' THEN 'Tata'
            WHEN fund_name LIKE '%Bandhan%' THEN 'Bandhan'
            WHEN fund_name LIKE '%Quant%' THEN 'Quant'
            WHEN fund_name LIKE '%HSBC%' THEN 'HSBC'
            WHEN fund_name LIKE '%Edelweiss%' THEN 'Edelweiss'
            WHEN fund_name LIKE '%Sundaram%' THEN 'Sundaram'
            ELSE 'Other'
        END as amc
    FROM holdings
    GROUP BY fund_name
)
SELECT 
    h.sector,
    fa.amc,
    ROUND(SUM(h.holding_pct), 2) as total_sector_weight,
    COUNT(DISTINCT h.stock_name) as stocks_in_sector
FROM holdings h
JOIN fund_amc fa ON h.fund_name = fa.fund_name
WHERE h.holding_pct > 0
GROUP BY h.sector, fa.amc
ORDER BY h.sector, total_sector_weight DESC
"""

result = pd.read_sql_query(query, conn)
# Show top AMC per sector
top_per_sector = result.groupby('sector').first().reset_index()
print(top_per_sector[['sector', 'amc', 'total_sector_weight']].head(20).to_string(index=False))

                                                                 sector           amc  total_sector_weight
                                                           2/3 wheelers         ICICI                15.20
                                                   Abrasives & bearings           SBI                 6.13
                                           Advertising & media agencies        Nippon                 0.01
                                                    Aerospace & defense         Kotak                10.28
                                                                Airline          HDFC                 5.76
                                             Airport & airport services  Aditya Birla                 1.04
                                                              Aluminium           SBI                 4.43
                                      Aluminium, copper & zinc products       Bandhan                 0.24
                                     

### Large Cap Overlap Matrix — how similar are large cap funds to each other?
A detailed pairwise comparison of all 10 large cap funds. This is the data you'd need to decide: "Should I switch from one large cap fund to another, or are they basically the same?"

In [23]:
query = """
SELECT 
    a.fund_name as fund_1,
    b.fund_name as fund_2,
    COUNT(DISTINCT a.stock_name) as common_stocks,
    ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as overlap_pct
FROM holdings a
JOIN holdings b 
    ON a.stock_name = b.stock_name
    AND a.fund_name < b.fund_name
WHERE a.holding_pct > 0 AND b.holding_pct > 0
    AND a.fund_name LIKE '%Large Cap%'
    AND b.fund_name LIKE '%Large Cap%'
GROUP BY a.fund_name, b.fund_name
ORDER BY overlap_pct DESC
"""

result = pd.read_sql_query(query, conn)
print("=== LARGE CAP OVERLAP MATRIX ===")
print(result.to_string(index=False))

=== LARGE CAP OVERLAP MATRIX ===
                              fund_1                          fund_2  common_stocks  overlap_pct
                 Axis Large Cap Fund    Canara Robeco Large Cap Fund             40        72.98
        Canara Robeco Large Cap Fund            Kotak Large Cap Fund             31        67.83
Aditya Birla Sun Life Large Cap Fund    Canara Robeco Large Cap Fund             38        66.98
        Canara Robeco Large Cap Fund     Nippon India Large Cap Fund             35        66.17
        Canara Robeco Large Cap Fund              UTI Large Cap Fund             35        65.79
Aditya Birla Sun Life Large Cap Fund            Kotak Large Cap Fund             33        64.48
                 Axis Large Cap Fund            Kotak Large Cap Fund             30        63.78
        Canara Robeco Large Cap Fund      Mirae Asset Large Cap Fund             32        62.88
Aditya Birla Sun Life Large Cap Fund      Mirae Asset Large Cap Fund             35        62.

### Mid-conviction stocks — the "sweet spot" between popular and hidden
Not the mega popular stocks (held by 25+ funds) and not the hidden gems (held by 1-3). These stocks are held by 5-15 funds with decent weight — popular enough to have fund manager consensus, but not so crowded that everyone owns them.

In [24]:
query = """
SELECT 
    stock_name,
    sector,
    COUNT(DISTINCT fund_name) as num_funds,
    ROUND(AVG(holding_pct), 2) as avg_weight,
    ROUND(SUM(holding_pct), 2) as total_weight,
    GROUP_CONCAT(DISTINCT market_cap) as cap_categories
FROM holdings
WHERE holding_pct > 0
GROUP BY stock_name
HAVING num_funds >= 5 AND num_funds <= 15
ORDER BY avg_weight DESC
LIMIT 20
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

                                 stock_name                               sector  num_funds  avg_weight  total_weight cap_categories
                    Nifty 50 : Futures Near           Exchange and data platform          5        3.85         19.26          Other
                Kalyan Jewellers India Ltd.          Gems, jewellery and watches          5        3.85         19.25          Other
                     Fortis Healthcare Ltd.                             Hospital          8        2.93         23.41          Other
                          Federal Bank Ltd.                  Private sector bank         12        2.63         31.57        Mid Cap
                                Indian Bank                   Public sector bank          7        2.40         16.80        Mid Cap
                        KEI Industries Ltd.                 Cables - electricals          8        2.34         18.70      Small Cap
                               Coforge Ltd.    Computers - software &

### Diversification Score — which fund is the most unique?
For each fund, I calculated its average overlap with every other fund in the database. Lower score = more unique portfolio. This helps identify truly differentiated funds vs. funds that look like index clones.

In [25]:
query = """
WITH pair_overlaps AS (
    SELECT 
        a.fund_name,
        b.fund_name as other_fund,
        ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as overlap
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name != b.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
    GROUP BY a.fund_name, b.fund_name
)
SELECT 
    fund_name,
    ROUND(AVG(overlap), 2) as diversification_score
FROM pair_overlaps
GROUP BY fund_name
ORDER BY diversification_score ASC
LIMIT 15
"""

result = pd.read_sql_query(query, conn)
print("Lower score = More unique fund")
print(result.to_string(index=False))

Lower score = More unique fund
                    fund_name  diversification_score
           SBI Small Cap Fund                   4.66
          HDFC Small Cap Fund                   4.93
       Bandhan Small Cap Fund                   4.99
           DSP Small Cap Fund                   5.15
         Quant Small Cap Fund                   6.50
              SBI Midcap Fund                   7.48
Franklin India Small Cap Fund                   7.49
          Axis Small Cap Fund                   7.84
          HSBC Small Cap Fund                   7.94
         Kotak Small Cap Fund                   7.94
            HDFC Mid Cap Fund                   9.59
      Mirae Asset Midcap Fund                   9.60
  Nippon India Small Cap Fund                  10.79
    Motilal Oswal Midcap Fund                  11.13
        Sundaram Mid Cap Fund                  11.46


### Market cap distribution across fund categories
Are "mid cap" funds actually investing in mid caps? Or are they sneaking in large caps for safety? This checks if funds are staying true to their SEBI-mandated category label.

In [26]:
query = """
WITH fund_cat AS (
    SELECT fund_name,
        CASE 
            WHEN fund_name LIKE '%Large Cap%' THEN 'Large Cap'
            WHEN fund_name LIKE '%Mid Cap%' OR fund_name LIKE '%Midcap%' THEN 'Mid Cap'
            WHEN fund_name LIKE '%Small Cap%' THEN 'Small Cap'
            WHEN fund_name LIKE '%Flexi%' OR fund_name LIKE '%Flexicap%' THEN 'Flexi Cap'
            WHEN fund_name LIKE '%Index%' OR fund_name LIKE '%Sensex%' OR fund_name LIKE '%Nifty%' THEN 'Index'
            ELSE 'Other'
        END as category
    FROM holdings
    GROUP BY fund_name
)
SELECT 
    fc.category,
    h.market_cap,
    COUNT(DISTINCT h.stock_name) as unique_stocks,
    ROUND(AVG(h.holding_pct), 2) as avg_weight,
    ROUND(SUM(h.holding_pct), 2) as total_weight
FROM holdings h
JOIN fund_cat fc ON h.fund_name = fc.fund_name
WHERE h.holding_pct > 0 AND h.market_cap != ''
GROUP BY fc.category, h.market_cap
ORDER BY fc.category, total_weight DESC
"""

result = pd.read_sql_query(query, conn)
print(result.to_string(index=False))

 category market_cap  unique_stocks  avg_weight  total_weight
Flexi Cap  Large Cap             60        2.14        437.14
Flexi Cap      Other            121        1.50        326.43
Flexi Cap    Mid Cap             56        1.17        116.11
Flexi Cap  Small Cap             61        0.76         57.83
    Index  Large Cap             33        2.29        352.38
    Index      Other             12        2.39        128.94
    Index    Mid Cap              5        0.84         18.47
Large Cap  Large Cap             63        1.99        622.50
Large Cap      Other             64        1.46        259.00
Large Cap    Mid Cap             44        0.79         69.25
Large Cap  Small Cap             23        0.59         17.05
  Mid Cap    Mid Cap             80        1.50        387.44
  Mid Cap      Other             95        1.29        315.72
  Mid Cap  Small Cap             60        1.21        146.62
  Mid Cap  Large Cap             33        1.38        113.02
Small Ca

### Optimal 5-fund portfolio with minimum overlap
The final practical output — if you could only hold 5 funds (one from each category), which combination gives you maximum diversification? I picked the most "unique" fund from each category based on the diversification score.

In [27]:
query = """
WITH fund_cat AS (
    SELECT fund_name,
        CASE 
            WHEN fund_name LIKE '%Large Cap%' THEN 'Large Cap'
            WHEN fund_name LIKE '%Mid Cap%' OR fund_name LIKE '%Midcap%' THEN 'Mid Cap'
            WHEN fund_name LIKE '%Small Cap%' THEN 'Small Cap'
            WHEN fund_name LIKE '%Flexi%' OR fund_name LIKE '%Flexicap%' THEN 'Flexi Cap'
            WHEN fund_name LIKE '%Index%' OR fund_name LIKE '%Sensex%' OR fund_name LIKE '%Nifty%' THEN 'Index'
            ELSE 'Other'
        END as category
    FROM holdings
    GROUP BY fund_name
),
pair_overlaps AS (
    SELECT 
        a.fund_name,
        ROUND(SUM(MIN(a.holding_pct, b.holding_pct)), 2) as overlap
    FROM holdings a
    JOIN holdings b 
        ON a.stock_name = b.stock_name
        AND a.fund_name != b.fund_name
    WHERE a.holding_pct > 0 AND b.holding_pct > 0
    GROUP BY a.fund_name, b.fund_name
),
fund_scores AS (
    SELECT 
        fund_name,
        ROUND(AVG(overlap), 2) as avg_overlap_score
    FROM pair_overlaps
    GROUP BY fund_name
)
SELECT 
    fc.category,
    fs.fund_name,
    fs.avg_overlap_score,
    COUNT(DISTINCT h.stock_name) as total_stocks
FROM fund_scores fs
JOIN fund_cat fc ON fs.fund_name = fc.fund_name
JOIN holdings h ON fs.fund_name = h.fund_name AND h.holding_pct > 0
GROUP BY fc.category, fs.fund_name
ORDER BY fc.category, fs.avg_overlap_score ASC
"""

result = pd.read_sql_query(query, conn)
best = result.groupby('category').first().reset_index()
print("=== BEST 5-FUND PORTFOLIO (Minimum Overlap) ===\n")
print(best.to_string(index=False))

=== BEST 5-FUND PORTFOLIO (Minimum Overlap) ===

 category                    fund_name  avg_overlap_score  total_stocks
Flexi Cap Motilal Oswal Flexi Cap Fund              17.67            23
    Index   HDFC BSE Sensex Index Fund              35.29            30
Large Cap           SBI Large Cap Fund              27.43            45
  Mid Cap              SBI Midcap Fund               7.48            51
Small Cap           SBI Small Cap Fund               4.66            67



# Key Findings

### 1. Index Funds Are Nearly Identical
Nifty 50 index funds from HDFC, SBI, UTI share 100% portfolio overlap. ICICI Prudential Nifty 50 overlaps 99.76% with them. Holding more than one Nifty 50 index fund is completely pointless.

### 2. Large Cap Funds Have 57% Average Overlap
More than half the portfolio is the same across large cap funds. This happens because SEBI mandates 80%+ investment in just ~100 stocks — fund managers have very little room to be different.

### 3. Small Cap Funds Are the Most Unique
Only 12% average overlap across small cap funds. With 1000+ stocks to choose from, each fund manager builds a genuinely different portfolio. Best category for true diversification.

### 4. ICICI Bank and HDFC Bank Dominate Indian Mutual Funds
ICICI Bank appears in 30 out of 45 funds with 6.61% average weight. HDFC Bank appears in 29 funds with 7.30% average weight. Between them they account for 9.5% of all equity weight in the dataset. Most mutual fund investors are heavily exposed to banking sector risk without realizing it.

### 5. Eternal Ltd. (Zomato) Is the Most Widely Held Stock
Found in 32 out of 45 funds — more than HDFC Bank or Reliance. The new-age tech stock has become a mutual fund darling across all categories.

### 6. Private Sector Banking Is the Most Crowded Sector
15.5% of all equity weight across the 45 funds — nearly 3x the second most crowded sector (IT at 5.6%). A banking crisis would hit almost every mutual fund investor in India simultaneously.

### 7. Index Funds Have Zero Small Cap Exposure
Nifty 50 and Sensex trackers contain absolutely no small cap stocks. Investors relying solely on index funds are completely missing out on the small cap segment of the market.

### 8. Holding Two Large Cap Funds Gives You 18% in Just Two Stocks
Split your money equally between HDFC Large Cap and ICICI Large Cap and 9.5% of it lands in ICICI Bank and 8.9% in HDFC Bank — 18.3% of your total money in two banking stocks. The two funds' weights add to 18.9% and 17.7%, but those are percentages of different funds: with equal money in each, your exposure is their average, not their sum.

### 9. Parag Parikh Flexi Cap Is the Most Unique Active Fund
Shows up in the least overlapping pairs consistently — its foreign holdings (Alphabet, Meta, Amazon, Microsoft) ensure minimal overlap with any other Indian equity fund. No other fund in our database holds US tech stocks.

### 10. Motilal Oswal Midcap and SBI Midcap Have Zero Overlap
Despite both being mid cap funds, they share absolutely no common stocks. Motilal Oswal runs a 26-stock high-conviction portfolio while SBI holds 51 stocks — proof that same category doesn't mean same portfolio.

### 11. Flexi Cap Funds Aren't Really "Flexi"
Flexi Cap funds put 46.6% of their equity in large caps vs 6.2% in small caps — a 7.6x skew. Despite being allowed to invest anywhere, fund managers overwhelmingly prefer large caps.

### 12. Don't Pair Large Cap with Index Funds
63.2% average overlap — the worst category combination. Large cap funds essentially hold the same stocks as Nifty 50, so holding both adds very little diversification.

### 13. Best Diversification Strategy
Pair a Small Cap fund with an Index, Large Cap or Flexi Cap fund — 3.9-4.7% average overlap, with Index + Small Cap the lowest at 3.9%. You get stability from the large-cap side plus genuine diversification from small caps.

### 14. The Optimal 5-Fund Portfolio
For maximum diversification, hold one fund from each category with the lowest overlap score:
- **Flexi Cap:** Motilal Oswal Flexi Cap Fund (23 stocks)
- **Index:** HDFC BSE Sensex Index Fund (30 stocks)
- **Large Cap:** SBI Large Cap Fund (45 stocks)
- **Mid Cap:** SBI Midcap Fund (51 stocks)
- **Small Cap:** SBI Small Cap Fund (67 stocks)

### 15. Concentration Reality Check — How Many Stocks Make Up Half the Portfolio?
HDFC BSE Sensex Index Fund needs just **6 stocks** to reach 50% of its portfolio, while Nippon India Small Cap Fund needs **59 stocks**. Motilal Oswal Flexi Cap and Motilal Oswal Midcap need only **8 stocks each** — confirming their high-conviction concentrated strategy. Bandhan Small Cap needs **47 stocks** to reach 50%, showing genuine diversification across its holdings.

## Limitations
- Data sourced from Moneycontrol.com which may have slight discrepancies with official AMC portfolio disclosures
- Holdings are as of a single date (Feb/Mar 2026) — overlap changes monthly as fund managers rebalance their portfolios
- Analysis covers equity holdings only — debt, cash, and other positions are excluded
- "Other" market cap category includes foreign stocks, recently listed companies, and boundary cases not classified by Moneycontrol

## Tools & Technologies Used
- **Python** — requests, BeautifulSoup (web scraping), pandas (data manipulation)
- **SQL (SQLite)** — database design, complex queries with JOINs, CTEs, window functions, aggregations
- **Data Source** — Moneycontrol.com (publicly available mutual fund portfolio holdings)